# 🎵 AI Music & Soundscape Generator — Training Notebook
### InternGrow AI Track — Task 3

This notebook trains an **LSTM** deep learning model on preprocessed MIDI musical sequences to learn musical patterns, then uses the trained model to generate new music.

**How to run:** Open this file in [Google Colab](https://colab.research.google.com), then run each cell in order (Runtime → Run all).

## Step 1: Install required libraries

In [1]:
!pip install music21 tensorflow numpy -q

## Step 2: Download a sample MIDI dataset

We use a small public-domain classical piano MIDI collection so this runs quickly. You can swap this for any MIDI folder of your choice — just point `midi_folder` at it.

In [2]:
import os

os.makedirs('midi_dataset', exist_ok=True)

# In Colab: click the folder icon in the left sidebar, open 'midi_dataset',
# and drag-and-drop your .mid files into it. A free dataset you can search for
# is the 'Classical Piano MIDI' dataset on Kaggle, or any small MIDI collection.
midi_folder = 'midi_dataset'
print('Upload your .mid files into the', midi_folder, 'folder (left sidebar), then continue.')

Upload your .mid files into the midi_dataset folder (left sidebar), then continue.


## Step 3: Preprocess MIDI files into note/chord sequences

This is the **preprocessing** step required by the task — converting raw MIDI into a format the neural network can learn from.

In [3]:
from music21 import converter, instrument, note, chord
import glob

def extract_notes(folder):
    notes = []
    for file in glob.glob(f'{folder}/*.mid'):
        midi = converter.parse(file)
        parts = instrument.partitionByInstrument(midi)
        elements = parts.parts[0].recurse() if parts else midi.flat.notes
        for el in elements:
            if isinstance(el, note.Note):
                notes.append(str(el.pitch))
            elif isinstance(el, chord.Chord):
                notes.append('.'.join(str(n) for n in el.normalOrder))
    return notes

notes = extract_notes(midi_folder)
print(f'Extracted {len(notes)} notes/chords from the dataset')

Extracted 553 notes/chords from the dataset


## Step 4: Encode notes into numeric sequences

Neural networks work with numbers, not note names — so we build a vocabulary and convert each note into an integer, then group them into fixed-length input sequences.

In [4]:
import numpy as np

pitch_names = sorted(set(notes))
note_to_int = {n: i for i, n in enumerate(pitch_names)}
int_to_note = {i: n for i, n in enumerate(pitch_names)}

sequence_length = 50
network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    seq_in = notes[i:i + sequence_length]
    seq_out = notes[i + sequence_length]
    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

n_vocab = len(pitch_names)
print(f'Vocabulary size: {n_vocab}, Training sequences: {len(network_input)}')

Vocabulary size: 41, Training sequences: 503


## Step 5: Build the LSTM deep learning architecture

This is the core **deep learning model** the task requires — an LSTM network that learns the patterns in musical sequences.

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=n_vocab, output_dim=64, input_length=sequence_length),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_vocab, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 6: Train the model

The model learns to predict the *next* note given a sequence of previous notes — this is how it learns musical patterns.

In [6]:
X = np.array(network_input)
y = np.array(network_output)

model.fit(X, y, epochs=40, batch_size=64)
model.save('music_lstm_model.h5')
print('Training complete. Model saved as music_lstm_model.h5')

Epoch 1/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 592ms/step - loss: 3.6381
Epoch 2/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 608ms/step - loss: 3.3923
Epoch 3/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 543ms/step - loss: 3.3347
Epoch 4/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 718ms/step - loss: 3.2873
Epoch 5/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 539ms/step - loss: 3.2748
Epoch 6/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 534ms/step - loss: 3.2166
Epoch 7/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 739ms/step - loss: 3.1560
Epoch 8/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 531ms/step - loss: 3.1237
Epoch 9/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 621ms/step - loss: 3.0633
Epoch 10/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 570ms/step - loss: 3.0400
Epoch 11/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 527ms/step - loss: 2.9702
Epoch 12/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 747ms/step - loss: 2.9180
Epoch 13/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 532ms/step - loss: 2.8689
Epoch 14/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 530ms/step - loss: 2.8187
Epoch 15/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 651ms/step - loss: 2.7846
Epoch 16/40
8/8 ━━

Training complete. Model saved as music_lstm_model.h5


## Step 7: Generate new music from the trained model

We feed the model a random starting sequence and let it predict new notes one at a time, building an original sequence.

In [7]:
import random

start = random.randint(0, len(network_input) - 1)
pattern = network_input[start]
generated_notes = []

for _ in range(200):
    input_seq = np.reshape(pattern, (1, sequence_length))
    prediction = model.predict(input_seq, verbose=0)
    idx = np.argmax(prediction)
    generated_notes.append(int_to_note[idx])
    pattern = pattern[1:] + [idx]

print('Generated', len(generated_notes), 'new notes')

Generated 200 new notes


## Step 8: Convert generated notes back into a MIDI file

In [8]:
from music21 import stream

output_stream = stream.Stream()
offset = 0

for item in generated_notes:
    if '.' in item:
        chord_notes = [note.Note(int(n)) for n in item.split('.')]
        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_stream.append(new_chord)
    else:
        new_note = note.Note(item)
        new_note.offset = offset
        output_stream.append(new_note)
    offset += 0.5

output_stream.write('midi', fp='generated_music.mid')
print('Saved generated_music.mid — download it from the Colab file browser (left sidebar)')

Saved generated_music.mid — download it from the Colab file browser (left sidebar)


## ✅ Deliverables for submission

1. This notebook (with outputs visible — run all cells before saving)
2. `generated_music.mid` — sample AI-generated music
3. `music_lstm_model.h5` — the trained model file
4. The mood-based soundscape web app (separate `index.html` file, upgrade feature)